# 03 · RAG pipeline

```
 question ─────────────────────────────────────────────┐
    │                                                  │ (keywords)
    ▼                                                  ▼
 (A) HyDE query parsing (LLM)                    ┌──────────────────────┐
     • hypothetical abstract ──► bi-encoder ───► │ (B) Weaviate hybrid  │
     • filters (years, categories)  ──────────►  │  BM25 + vector + filt│──► top 50 chunks
                                                 └──────────────────────┘
                                                            │
                                                            ▼
                                    (C) cross-encoder re-ranker (bge-reranker-v2-m3)
                                        best chunk per paper ──► top 6 papers
                                                            │
                                                            ▼
                                    (D) LLM answer, ONLY from those papers, cited [1]..[6]
                                        (abstains when evidence is too weak)
```

All models run locally and stay in GPU memory for the lifetime of the kernel:

| Role | Model | Library |
|---|---|---|
| Bi-encoder | `BAAI/bge-large-en-v1.5` | sentence-transformers |
| Re-ranker | `BAAI/bge-reranker-v2-m3` | sentence-transformers |
| LLM | `Qwen/Qwen3.5-9B` (bf16) | transformers |

In [1]:
import json
import time

import numpy as np
import torch
from IPython.display import Markdown, display
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from weaviate.classes.query import Filter, HybridFusion, MetadataQuery

from common import *

embedder = SentenceTransformer(EMBED_MODEL, device="cuda").half()
reranker = CrossEncoder(RERANK_MODEL, device="cuda", max_length=384)
reranker.model.half()

client = get_weaviate_client()
collection = client.collections.get(COLLECTION)
n_objects = collection.aggregate.over_all(total_count=True).total_count
print(f"Weaviate: {n_objects:,} chunks in '{COLLECTION}'")

# The LLM: ~18 GB in bf16, loads in ~15 s from the local Hugging Face cache.
llm_tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype=torch.bfloat16, device_map="cuda:0")
print(f"LLM: {LLM_MODEL} | GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Weaviate: 257,025 chunks in 'ArxivChunk'


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

LLM: Qwen/Qwen3.5-9B | GPU memory in use: 19.7 GB


## LLM helper
`llm_chat()` applies the Qwen chat template with `enable_thinking=False` (skips the reasoning phase),
generates greedily (or samples when `temperature > 0`), and in `json_mode` returns only the outermost
JSON object. The first call takes ~1 min while GPU kernels compile; generation then runs at ~29 tokens/s.

In [2]:
def llm_chat(messages, temperature=0.2, max_tokens=800, json_mode=False):
    if json_mode:
        last = messages[-1]
        messages = messages[:-1] + [{**last, "content": last["content"] + "\n\nRespond with a single JSON object only."}]
    prompt = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
    inputs = llm_tok(prompt, return_tensors="pt").to(llm.device)
    sampling = {"do_sample": True, "temperature": temperature, "top_p": 0.9} if temperature > 0 else {"do_sample": False}
    with torch.inference_mode():
        out = llm.generate(**inputs, max_new_tokens=max_tokens, **sampling)
    text = llm_tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    if json_mode:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end > start:
            text = text[start:end + 1]
    return text

print(llm_chat([{"role": "user", "content": "Reply with exactly: ready"}], temperature=0))   # also warms up the GPU kernels

ready


## (A) Query parsing with HyDE
Questions are short and phrased differently from the abstracts that answer them. Following HyDE
(Gao et al., 2022), the LLM writes a hypothetical abstract for the question, and that abstract's
embedding drives the vector search. The abstract is never shown to the user, so factual errors in it
are harmless.

A single LLM call returns JSON with:
- `hypothetical_abstract`
- `year_from`, `year_to`: set only when the question restricts the time range
- `categories`: `cs.CV` / `cs.CL` only when the question clearly belongs to one field

The search vector is the normalised mean of the question embedding (with BGE's query instruction) and
the abstract embedding; keeping the question in it limits drift when the abstract goes off-topic.
BM25 uses the original question, so exact terms ("LoRA", "KV cache") match literally.

In [3]:
PARSE_SYSTEM = """You are the query parser of a search engine over arXiv papers in computer vision (cs.CV) and NLP (cs.CL), submitted between {start} and {end}.
Given a user question, return a JSON object with exactly these keys:
- "hypothetical_abstract": a plausible 120-180 word arXiv abstract of a paper that would answer the question, written in the typical style of the field (problem, method, results). Use the technical terminology a real paper would use.
- "year_from": integer or null. Only set if the user explicitly restricts the time range (e.g. "since 2025", "recent work in 2026").
- "year_to": integer or null. Same rule.
- "categories": list drawn from ["cs.CV", "cs.CL"]. Use ["cs.CV"] only if the question is clearly pure vision, ["cs.CL"] only if clearly pure language. Use [] if it is multimodal, ambiguous or both.
Return only the JSON object."""

def parse_query(question):
    raw = llm_chat([{"role": "system", "content": PARSE_SYSTEM.format(start=START_DATE.year, end=END_DATE.year)},
                    {"role": "user", "content": question}],
                   temperature=0.3, max_tokens=500, json_mode=True)
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:                      # never let parsing break retrieval
        parsed = {}
    cats = [c for c in parsed.get("categories") or [] if c in CATEGORIES]
    to_int = lambda v: int(v) if isinstance(v, (int, float)) or (isinstance(v, str) and v.isdigit()) else None
    return {
        "hypothetical_abstract": parsed.get("hypothetical_abstract") or question,
        "year_from": to_int(parsed.get("year_from")),
        "year_to": to_int(parsed.get("year_to")),
        "categories": cats,
    }

def hyde_vector(question, hypothetical_abstract):
    q_vec, d_vec = embedder.encode([QUERY_INSTRUCTION + question, hypothetical_abstract],
                                   normalize_embeddings=True)
    v = q_vec + d_vec
    return (v / np.linalg.norm(v)).tolist()

demo_q = "How do recent papers reduce hallucination in vision-language models?"
t = time.time()
parsed = parse_query(demo_q)
print(f"parsed in {time.time() - t:.1f}s\n")
print(json.dumps(parsed, indent=2))

parsed in 9.5s

{
  "hypothetical_abstract": "Vision-language models (VLMs) frequently suffer from hallucinations, generating text that contradicts visual evidence or invents non-existent objects. This paper introduces \\textit{Contrastive Visual Grounding with Semantic Consistency (CVG-SC)}, a novel framework designed to mitigate these errors through dual-path alignment. First, we propose a dynamic attention masking mechanism that penalizes token generation probabilities for visual regions lacking sufficient semantic support, effectively suppressing confident but incorrect predictions. Second, we integrate a cross-modal consistency loss that enforces strict alignment between generated captions and salient visual features, ensuring textual descriptions remain grounded in observed reality. Extensive experiments on COCO, GQA, and TextVQA benchmarks demonstrate that CVG-SC reduces hallucination rates by 24.5% compared to state-of-the-art baselines like LLaVA-1.6 and InstructBLIP. Our meth

## (B) Hybrid search
One Weaviate `hybrid()` query:
- **BM25** over `title^2` and `text`: exact terms, acronyms, model names.
- **Vector search** with the HyDE vector: meaning and paraphrase.
- **Fusion:** `relativeScoreFusion`, i.e. min-max-normalised scores combined as
  `alpha·vector + (1−alpha)·bm25` (`alpha = 0.6`). Unlike rank-based fusion (RRF), it preserves score gaps.
- **Filters** are applied inside the search (pre-filtering).

50 candidates are retrieved; precision is left to the re-ranker.

In [4]:
ALPHA = 0.6
N_CANDIDATES = 50

def build_filters(year_from=None, year_to=None, categories=None):
    parts = []
    if year_from:
        parts.append(Filter.by_property("year").greater_or_equal(year_from))
    if year_to:
        parts.append(Filter.by_property("year").less_or_equal(year_to))
    if categories:
        parts.append(Filter.by_property("categories").contains_any(categories))
    return Filter.all_of(parts) if parts else None

def hybrid_search(question, vector, filters=None, alpha=ALPHA, limit=N_CANDIDATES):
    res = collection.query.hybrid(
        query=question,
        vector=vector,
        alpha=alpha,
        limit=limit,
        query_properties=["title^2", "text"],
        fusion_type=HybridFusion.RELATIVE_SCORE,
        filters=filters,
        return_metadata=MetadataQuery(score=True),
    )
    hits = []
    for o in res.objects:
        props = dict(o.properties)
        props["published"] = props["published"].strftime("%Y-%m")   # Weaviate returns DATE as datetime
        hits.append({**props, "hybrid_score": o.metadata.score})
    return hits

t = time.time()
candidates = hybrid_search(demo_q, 
                           hyde_vector(demo_q, parsed["hypothetical_abstract"]),
                           build_filters(parsed["year_from"], parsed["year_to"], parsed["categories"]))
print(f"{len(candidates)} candidates in {(time.time() - t) * 1000:.0f} ms")
for c in candidates[:5]:
    print(f"  {c['hybrid_score']:.3f}  [{c['arxiv_id']}] {c['title'][:90]}")

50 candidates in 223 ms
  0.600  [2511.05017] Towards Mitigating Hallucinations in Large Vision-Language Models by Refining Textual Embe
  0.521  [2505.14257] Mitigating Hallucination in Large Vision-Language Models through Aligning Attention Distri
  0.411  [2506.23590] CAI: Caption-Sensitive Attention Intervention for Mitigating Object Hallucination in Large
  0.400  [2604.22822] DO-Bench: An Attributable Benchmark for Diagnosing Object Hallucination in Vision-Language
  0.396  [2603.29405] Hallucination-aware intermediate representation edit in large vision-language models


## (C) Re-ranking

| | Bi-encoder (retrieval) | Cross-encoder (re-ranking) |
|---|---|---|
| Input | query and document encoded separately | `[query, document]` encoded together |
| Interaction | cosine similarity of two vectors | full attention across query and document tokens |
| Cost | documents pre-embedded, ms per query | one forward pass per (query, document) pair |
| Accuracy | good | considerably higher |

The cross-encoder re-scores the 50 candidates against the question (sigmoid scores in [0, 1]). Only the
best chunk of each paper is kept, so the 6 sources are distinct papers.

In [5]:
TOP_K_PAPERS = 6

def one_per_paper(ranked):
    """Keep only the first (= best-ranked) chunk of each paper, preserving order."""
    best = {}
    for c in ranked:
        best.setdefault(c["arxiv_id"], c)
    return list(best.values())

def rerank(question, candidates, top_k=TOP_K_PAPERS):
    if not candidates:
        return []
    pairs = [(question, f"{c['title']}\n{c['text']}") for c in candidates]
    scores = reranker.predict(pairs, batch_size=64, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return one_per_paper(sorted(candidates, key=lambda c: -c["rerank_score"]))[:top_k]

t = time.time()
top = rerank(demo_q, candidates)
print(f"re-ranked in {(time.time() - t) * 1000:.0f} ms\n")
for c in top:
    print(f"  rerank {c['rerank_score']:.3f} | hybrid {c['hybrid_score']:.3f} | {c['title'][:85]}")

re-ranked in 123 ms

  rerank 0.972 | hybrid 0.375 | VaLiD: Mitigating the Hallucination of Large Vision Language Models by Visual Layer F
  rerank 0.967 | hybrid 0.393 | Mitigating Hallucinations in Vision-Language Models through Image-Guided Head Suppres
  rerank 0.962 | hybrid 0.396 | Hallucination-aware intermediate representation edit in large vision-language models
  rerank 0.962 | hybrid 0.135 | VCE: A zero-cost hallucination mitigation method of LVLMs via visual contrastive edit
  rerank 0.960 | hybrid 0.191 | Hallucinations Leave a Grounding Signature:Verifier-Guided Decoding for Selective Obj
  rerank 0.956 | hybrid 0.346 | VEGAS: Mitigating Hallucinations in Large Vision-Language Models via Vision-Encoder A


## (D) Grounded generation
The LLM answers from the numbered sources only, cites every claim (`[2]`), compares approaches rather
than summarising papers one by one, and states what is missing when the sources are insufficient.

If the best re-ranker score is below `MIN_RERANK_SCORE`, the pipeline abstains without calling the LLM.

In [6]:
MIN_RERANK_SCORE = 0.05

ANSWER_SYSTEM = """You are ArxivSage, a research assistant for computer vision and NLP.
Answer the user's question using ONLY the numbered arXiv sources provided.
Rules:
- Cite every factual claim with the source number in square brackets, e.g. [1] or [2][4].
- Synthesize: group papers by approach and compare them, rather than listing them one by one.
- Be concise: 1 short intro sentence, then 3-6 bullet points, then a one-line takeaway.
- If the sources do not contain enough information to answer, say exactly what is missing. Never use outside knowledge and never invent papers or numbers."""

def build_context(papers):
    blocks = []
    for i, p in enumerate(papers, 1):
        blocks.append(f"[{i}] {p['title']} (arXiv:{p['arxiv_id']}, {p['published']}, {p['primary_category']})\n{p['text']}")
    return "\n\n".join(blocks)

def generate_answer(question, papers):
    if not papers or papers[0]["rerank_score"] < MIN_RERANK_SCORE:
        return ("I couldn't find papers in the indexed CV/NLP arXiv corpus "
                f"({START_DATE.year}–{END_DATE.year}) that address this question.")
    user = f"Sources:\n\n{build_context(papers)}\n\nQuestion: {question}"
    return llm_chat([{"role": "system", "content": ANSWER_SYSTEM},
                     {"role": "user", "content": user}], temperature=0.2, max_tokens=700)

## (E) `ask()`
End-to-end pipeline with per-stage timings. Explicit `year_from` / `year_to` / `categories` arguments
override the filters inferred by the LLM; `use_hyde` and `use_rerank` disable components for the ablation.

In [7]:
def ask(question, year_from=None, year_to=None, categories=None, use_hyde=True, use_rerank=True, verbose=True):
    timings, t0 = {}, time.time()

    if use_hyde:
        parsed = parse_query(question)
        vector = hyde_vector(question, parsed["hypothetical_abstract"])
    else:
        parsed = {"hypothetical_abstract": None, "year_from": None, "year_to": None, "categories": []}
        vector = embedder.encode(QUERY_INSTRUCTION + question, normalize_embeddings=True).tolist()
    timings["parse+embed"] = time.time() - t0

    filters = build_filters(year_from or parsed["year_from"],
                            year_to or parsed["year_to"],
                            categories or parsed["categories"])
    t = time.time()
    candidates = hybrid_search(question, vector, filters)
    timings["hybrid_search"] = time.time() - t

    t = time.time()
    if use_rerank:
        papers = rerank(question, candidates)
    else:
        papers = one_per_paper(candidates)[:TOP_K_PAPERS]
        for p in papers:
            p["rerank_score"] = 1.0                   # no gate without a re-ranker
    timings["rerank"] = time.time() - t

    t = time.time()
    answer = generate_answer(question, papers)
    timings["generate"] = time.time() - t
    timings["total"] = time.time() - t0

    result = {"question": question, "answer": answer, "sources": papers, "parsed": parsed, "timings": timings}
    if verbose:
        show(result)
    return result

def show(result):
    md = [f"### Q: {result['question']}", result["answer"]]
    abstained = not result["sources"] or result["sources"][0]["rerank_score"] < MIN_RERANK_SCORE
    if not abstained:
        md.append("\n**Sources**")
    for i, p in enumerate([] if abstained else result["sources"], 1):
        md.append(f"{i}. [{p['title']}]({p['url']}) · {p['published']} · {p['primary_category']} "
                  f"· rerank={p['rerank_score']:.2f}")
    md.append("\n*" + " · ".join(f"{k} {v:.1f}s" for k, v in result["timings"].items()) + "*")
    display(Markdown("\n\n".join(md)))

## Examples

In [8]:
_ = ask("How do recent papers reduce hallucination in vision-language models?")

### Q: How do recent papers reduce hallucination in vision-language models?

Recent papers reduce hallucination in vision-language models by employing inference-time interventions that modify attention mechanisms, decoding strategies, or internal representations without requiring retraining.

*   **Uncertainty-Guided Re-attention:** Methods like the one in [1] use frozen, pretrained models to iteratively refine responses by quantifying multidimensional uncertainty (e.g., token entropy, attention dispersion) and cropping under-explored regions to reduce hallucination rates by 9.8 percentage points [1].
*   **Visual Encoding Adjustments:** Approaches such as VaLiD [2] and VCE [5] address distortions in visual layers or language priors by applying contrastive decoding or visual contrastive editing to suppress hallucinatory tendencies during inference [2][5].
*   **Attention Head Suppression:** SPIN [3] identifies that hallucinations stem from specific dynamic subsets of attention heads and suppresses them via an image-guided strategy that incurs no significant latency overhead [3].
*   **Intermediate Representation Editing:** Frameworks like the one in [4] dynamically detect hallucination representations and perform edits on these intermediate states to eliminate errors with minimal computational cost [4].
*   **Preference Optimization:** Some methods utilize Direct Preference Optimization (DPO) with "hard grounding" preference pairs to supervise the model on fine-grained recognition and evidence utilization, focusing on cases where visual support is ambiguous or insufficient [6].

**Takeaway:** Recent strategies predominantly focus on training-free, inference-time modifications to attention, decoding, or internal representations to align model outputs with visual evidence.


**Sources**

1. [Toward More Reliable Artificial Intelligence: Reducing Hallucinations in Vision-Language Models](https://arxiv.org/abs/2512.07564) · 2025-12 · cs.CV · rerank=0.98

2. [VaLiD: Mitigating the Hallucination of Large Vision Language Models by Visual Layer Fusion Contrastive Decoding](https://arxiv.org/abs/2411.15839) · 2024-11 · cs.CV · rerank=0.97

3. [Mitigating Hallucinations in Vision-Language Models through Image-Guided Head Suppression](https://arxiv.org/abs/2505.16411) · 2025-05 · cs.CV · rerank=0.97

4. [Hallucination-aware intermediate representation edit in large vision-language models](https://arxiv.org/abs/2603.29405) · 2026-03 · cs.CV · rerank=0.96

5. [VCE: A zero-cost hallucination mitigation method of LVLMs via visual contrastive editing](https://arxiv.org/abs/2604.19412) · 2026-04 · cs.CV · rerank=0.96

6. [Reducing Hallucination in Multimodal Large Language Models through Hard Grounding Preference Supervision](https://arxiv.org/abs/2605.16411) · 2026-05 · cs.CV · rerank=0.96


*parse+embed 9.2s · hybrid_search 0.0s · rerank 0.1s · generate 11.6s · total 20.9s*

In [9]:
_ = ask("What is the recent progress in pedestrian attribute recognition in low light conditions?")

### Q: What is the recent progress in pedestrian attribute recognition in low light conditions?

Recent progress in pedestrian attribute recognition (PAR) for low-light conditions focuses on leveraging event cameras, synthetic data, and unified multimodal frameworks to overcome RGB limitations like motion blur and poor visibility.

*   **RGB-Event Fusion:** New approaches utilize event cameras to provide motion cues that enhance RGB performance in low-light and high-speed scenarios, addressing the sensitivity of traditional RGB cameras [1].
*   **Efficient Event Processing:** To reduce computational overhead in RGB-Event systems, recent methods employ lightweight Discrete Cosine Transform (DCT) operations and associative memory-augmented learning to extract frequency-domain features that augment RGB branches [2].
*   **Synthetic Data Expansion:** Researchers are investigating image-to-image diffusion models to generate diverse synthetic pedestrian images, aiming to improve model robustness and adaptability in complex, low-light environments where annotated data is scarce [3].
*   **Unified Multimodal Frameworks:** Unified frameworks like UniPAR integrate heterogeneous modalities (RGB, video, event streams) with phased fusion encoders to align visual features with textual queries, demonstrating improved cross-domain generalization in extreme low-light conditions [6].
*   **Spectral Optimization for Detection:** While primarily focused on detection, methods like PedDet decouple pedestrian and background features using adaptive spectral optimization and contrastive alignment, achieving state-of-the-art accuracy in low-light conditions which supports downstream attribute tasks [5].

Recent advancements prioritize multimodal fusion with event cameras and synthetic data generation to significantly improve robustness and accuracy in low-light pedestrian attribute recognition.


**Sources**

1. [RGB-Event based Pedestrian Attribute Recognition: A Benchmark Dataset and An Asymmetric RWKV Fusion Framework](https://arxiv.org/abs/2504.10018) · 2025-04 · cs.CV · rerank=0.88

2. [PFM-VEPAR: Prompting Foundation Models for RGB-Event Camera based Pedestrian Attribute Recognition](https://arxiv.org/abs/2603.19565) · 2026-03 · cs.CV · rerank=0.86

3. [Enhancing Zero-Shot Pedestrian Attribute Recognition with Synthetic Data Generation: A Comparative Study with Image-To-Image Diffusion Models](https://arxiv.org/abs/2509.02161) · 2025-09 · cs.CV · rerank=0.83

4. [Adversarial Semantic and Label Perturbation Attack for Pedestrian Attribute Recognition](https://arxiv.org/abs/2505.23313) · 2025-05 · cs.CV · rerank=0.78

5. [PedDet: Adaptive Spectral Optimization for Multimodal Pedestrian Detection](https://arxiv.org/abs/2502.14063) · 2025-02 · cs.CV · rerank=0.72

6. [UniPAR: A Unified Framework for Pedestrian Attribute Recognition](https://arxiv.org/abs/2603.05114) · 2026-03 · cs.CV · rerank=0.63


*parse+embed 8.1s · hybrid_search 0.0s · rerank 0.1s · generate 11.1s · total 19.3s*

In [10]:
_ = ask("What approaches exist for visual question answering on remote sensing or satellite imagery?")

### Q: What approaches exist for visual question answering on remote sensing or satellite imagery?

Approaches for visual question answering (VQA) on remote sensing imagery range from generative Large Vision-Language Models (LVLMs) and chain-of-thought reasoning to parameter-efficient fine-tuning and multi-agent frameworks.

*   **Generative LVLMs** utilize a two-step strategy of domain-adaptive pretraining and prompt-based finetuning to generate natural language answers without predefined categories, addressing the limitations of separate feature extractors [1].
*   **Chain-of-Thought (CoT) Reasoning** integrates structured intermediate rationales with Direct Preference Optimization (DPO) to enhance interpretability and accuracy for complex geospatial tasks like disaster monitoring and spatial relation analysis [2].
*   **Parameter-Efficient Fine-Tuning (PEFT)** adapts general foundation models to remote sensing by applying strategies like RS Adapters across Dual Encoder, Encoder-Decoder, and Hybrid architectures to mitigate domain shifts and computational costs [3].
*   **Multi-Agent Reasoning** employs training-free frameworks that wrap frozen models with retrieval modules to handle long visual contexts, multi-scale tiles, and multi-temporal frames while ensuring answers are grounded in specific image evidence [4].
*   **Knowledge-Aware Question Generation** enriches simplistic, template-based queries by incorporating external knowledge triplets and commonsense knowledge, using image captioning as an intermediary to improve contextual understanding and image grounding [5][6].

The field is evolving from basic accuracy-focused models toward frameworks that prioritize structured reasoning, evidence grounding, and the integration of external knowledge to handle the complexity of satellite imagery.


**Sources**

1. [Large Vision-Language Models for Remote Sensing Visual Question Answering](https://arxiv.org/abs/2411.10857) · 2024-11 · cs.CV · rerank=1.00

2. [Geospatial Chain of Thought Reasoning for Enhanced Visual Question Answering on Satellite Imagery](https://arxiv.org/abs/2511.11198) · 2025-11 · cs.CV · rerank=0.99

3. [A Unified Framework for Efficient Remote Sensing Visual Question Answering: Adapting Dual, Hybrid, and Encoder-Decoder Architectures](https://arxiv.org/abs/2606.19277) · 2026-06 · cs.CV · rerank=0.94

4. [TerraAgent: Demonstration-Grounded Multi-Agent Reasoning for Long-Context Remote Sensing Visual Question Answering](https://arxiv.org/abs/2610.12517) · 2026-10 · cs.CV · rerank=0.93

5. [Knowledge-aware Visual Question Generation for Remote Sensing Images](https://arxiv.org/abs/2602.19224) · 2026-02 · cs.CV · rerank=0.92

6. [Questions beyond Pixels: Integrating Commonsense Knowledge in Visual Question Generation for Remote Sensing](https://arxiv.org/abs/2602.19217) · 2026-02 · cs.CV · rerank=0.91


*parse+embed 9.9s · hybrid_search 0.0s · rerank 0.1s · generate 11.4s · total 21.5s*

Time filter inferred from the question ("since 2026"):

In [11]:
r = ask("What new methods for KV cache compression in long-context LLMs were proposed since 2026?")
print("parsed filters:", {k: v for k, v in r["parsed"].items() if k != "hypothetical_abstract"})

### Q: What new methods for KV cache compression in long-context LLMs were proposed since 2026?

Since 2026, several novel training-free and optimization-based methods have been proposed to address KV cache compression in long-context LLMs, moving beyond traditional low-rank approximation and scalar quantization:

*   **Vector Quantization (VQ) Approaches:** VQKV [1] and TaSQ [3] introduce vector quantization to achieve high compression ratios (up to 82.8%) and extreme 1-bit compression respectively, replacing scalar quantization to better preserve fidelity and handle channel statistics [1][3].
*   **Global Regression and Anchoring:** GRKV [2] and AnchorKV [4] propose merging strategies that minimize attention output discrepancies or use anchor-residual representations to compress caches by up to 20× without discarding tokens, addressing the imbalance in span-based retention [2][4].
*   **Distillation and Gating:** KVSculpt [5] optimizes a smaller set of unconstrained KV pairs in continuous embedding space rather than selecting or combining original entries, while Fast KVzip [6] employs lightweight sink-attention gating modules for efficient eviction with negligible computational cost [5][6].

These methods collectively shift the paradigm from simple eviction or scalar quantization toward vector quantization, global regression, and continuous space optimization to balance memory efficiency with model performance.


**Sources**

1. [VQKV: High-Fidelity and High-Ratio Cache Compression via Vector-Quantization](https://arxiv.org/abs/2603.16435) · 2026-03 · cs.CL · rerank=0.99

2. [GRKV: Global Regression for Training-Free KV Cache Compression in Long-Context LLMs](https://arxiv.org/abs/2605.31105) · 2026-05 · cs.CL · rerank=0.97

3. [Tailoring the Quantization Space for 1-Bit KV Cache Compression](https://arxiv.org/abs/2610.03027) · 2026-10 · cs.LG · rerank=0.97

4. [AnchorKV: Anchor-Residual KV Cache Compression](https://arxiv.org/abs/2608.02901) · 2026-08 · cs.LG · rerank=0.96

5. [KVSculpt: KV Cache Compression as Distillation](https://arxiv.org/abs/2603.27819) · 2026-03 · cs.LG · rerank=0.96

6. [Fast KVzip: Efficient and Accurate LLM Inference with Gated KV Eviction](https://arxiv.org/abs/2601.17668) · 2026-01 · cs.LG · rerank=0.96


*parse+embed 9.7s · hybrid_search 0.0s · rerank 0.1s · generate 9.8s · total 19.6s*

parsed filters: {'year_from': 2026, 'year_to': None, 'categories': ['cs.CL']}


Out-of-scope question (abstains):

In [12]:
_ = ask("What is the best recipe for chocolate cake?")

### Q: What is the best recipe for chocolate cake?

I couldn't find papers in the indexed CV/NLP arXiv corpus (2024–2026) that address this question.


*parse+embed 7.0s · hybrid_search 0.0s · rerank 0.1s · generate 0.0s · total 7.1s*

## Ablation
Top-5 papers for one question with HyDE and re-ranking enabled and disabled.

In [13]:
import pandas as pd

def top_titles(question, use_hyde, use_rerank, k=5):
    if use_hyde:
        p = parse_query(question)
        vec = hyde_vector(question, p["hypothetical_abstract"])
    else:
        vec = embedder.encode(QUERY_INSTRUCTION + question, normalize_embeddings=True).tolist()
    cands = hybrid_search(question, vec)
    if use_rerank:
        cands = rerank(question, cands, top_k=k)
    else:
        cands = one_per_paper(cands)[:k]
    return [c["title"][:70] for c in cands]

ablation_q = "Which techniques make diffusion models generate images faster with fewer sampling steps?"
table = {f"hyde={h}, rerank={r}": top_titles(ablation_q, h, r)
         for h in (False, True) for r in (False, True)}
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(table, index=[f"#{i}" for i in range(1, 6)])

,"hyde=False, rerank=False","hyde=False, rerank=True","hyde=True, rerank=False","hyde=True, rerank=True"
#1,Glance: Accelerating Diffusion Models with 1 Sample,Accelerating Video Diffusion Models via Distribution Matching,Inference-Time Scaling for Diffusion Models beyond Scaling Denoising S,Accelerating Video Diffusion Models via Distribution Matching
#2,Optimize Your Sampling: Tuned Diffusion Sampling with Bayesian Optimiz,LowDiff: Efficient Diffusion Sampling with Low-Resolution Condition,Glance: Accelerating Diffusion Models with 1 Sample,LowDiff: Efficient Diffusion Sampling with Low-Resolution Condition
#3,Inference-Time Scaling for Diffusion Models beyond Scaling Denoising S,One-Way Ticket:Time-Independent Unified Encoder for Distilling Text-to,UnStep: Training-Free Acceleration of Causal Video Diffusion with Fewe,Learning Few-Step Diffusion Models by Trajectory Distribution Matching
#4,UnStep: Training-Free Acceleration of Causal Video Diffusion with Fewe,Schedule On the Fly: Diffusion Time Prediction for Faster and Better I,Breaking the Likelihood-Quality Trade-off in Diffusion Models by Mergi,One-Way Ticket:Time-Independent Unified Encoder for Distilling Text-to
#5,DiSA: Diffusion Step Annealing in Autoregressive Image Generation,Pretrained Diffusion Models Are Inherently Skipped-Step Samplers,LowDiff: Efficient Diffusion Sampling with Low-Resolution Condition,Parallel Decoding Distillation for Fast Image and Video Generation


## Custom query
Optional filters: `year_from`, `year_to`, `categories=["cs.CV"]` or `["cs.CL"]`.

In [14]:
_ = ask("What are the latest benchmarks for evaluating multimodal LLM reasoning?", categories=["cs.CV"])

### Q: What are the latest benchmarks for evaluating multimodal LLM reasoning?

Recent benchmarks for evaluating multimodal LLM reasoning focus on expanding domain coverage, refining reasoning types, and addressing specific visual challenges like dynamic scenes and multi-image inputs.

*   **Domain and Academic Scope:** MDK12-Bench evaluates reasoning across six K-12 disciplines (math, physics, chemistry, biology, geography, and information science) using 140K real-world exam instances [1].
*   **Logical Reasoning Types:** MME-Reasoning specifically targets logical reasoning by covering inductive, deductive, and abductive types while filtering out perceptual skills and knowledge breadth [3].
*   **Multi-Image Processing:** MMRB is the first benchmark designed to evaluate structured visual reasoning across multiple images, featuring 92 sub-tasks covering spatial, temporal, and semantic reasoning [2].
*   **Dynamic and Continuous Scenes:** ViSTR-Bench assesses qualitative reasoning from continuous visual cues in dynamic scenes, moving beyond static scenes or exact quantitative predictions [4].
*   **Visual Fidelity and Explanation:** VERIFY isolates visual reasoning capabilities by providing minimal textual context and includes human-annotated reasoning paths to evaluate model decision-making [5].
*   **Visual Diversity:** WorldBench introduces a taxonomy of thousands of visual concepts to ensure models are tested on open-ended, visually diverse inputs rather than just expanding task types [6].

These benchmarks collectively shift evaluation from simple recognition toward complex, domain-specific, and visually diverse reasoning tasks.


**Sources**

1. [MDK12-Bench: A Multi-Discipline Benchmark for Evaluating Reasoning in Multimodal Large Language Models](https://arxiv.org/abs/2504.05782) · 2025-04 · cs.CV · rerank=0.99

2. [Evaluating MLLMs with Multimodal Multi-image Reasoning Benchmark](https://arxiv.org/abs/2506.04280) · 2025-06 · cs.CV · rerank=0.98

3. [MME-Reasoning: A Comprehensive Benchmark for Logical Reasoning in MLLMs](https://arxiv.org/abs/2505.21327) · 2025-05 · cs.AI · rerank=0.98

4. [ViSTR-Bench: Can MLLMs Reason from Continuous Visual Cues in Dynamic Scenes?](https://arxiv.org/abs/2607.20868) · 2026-07 · cs.CV · rerank=0.98

5. [VERIFY: A Benchmark of Visual Explanation and Reasoning for Investigating Multimodal Reasoning Fidelity](https://arxiv.org/abs/2503.11557) · 2025-03 · cs.CV · rerank=0.97

6. [WorldBench: A Challenging and Visually Diverse Multimodal Reasoning Benchmark](https://arxiv.org/abs/2606.06538) · 2026-06 · cs.CV · rerank=0.97


*parse+embed 9.8s · hybrid_search 0.0s · rerank 0.1s · generate 10.8s · total 20.7s*